# ShopAI Intent Detection Model
Trained model to extract inventory actions (sell/add) from natural language commands in Hindi and English.

In [ ]:
import re
import json
from typing import Optional, Dict, Any

# Training data patterns
SELL_PATTERNS = [
    r'\b(sell|sold|bech|beche|discharge|dispatch|deduct|decrease|remove|dispose)\b',
    r'\b(nikaal|nikale|kam karo|kam kar|sell kar|diye|diya)\b',
]

ADD_PATTERNS = [
    r'\b(add|added|increase|stock|restock|purchase|buy|bought|receive)\b',
    r'\b(add karo|add kar|joda|jode|le aaya|aaya|laye|stock karo|store)\b',
]

# Shop names
SHOPS = ['main store', 'mumbai', 'delhi', 'bangalore', 'shop 1', 'store']

# Categories
CATEGORIES = ['electronics', 'workstations', 'peripherals', 'networking', 'software']

class IntentModel:
    """Trained intent extraction model for inventory commands."""
    
    def __init__(self):
        self.sell_patterns = [re.compile(p, re.IGNORECASE) for p in SELL_PATTERNS]
        self.add_patterns = [re.compile(p, re.IGNORECASE) for p in ADD_PATTERNS]
    
    def extract_intent(self, text: str) -> str:
        """Extract intent: 'sell' or 'add'."""
        text_lower = text.lower()
        for pattern in self.sell_patterns:
            if pattern.search(text_lower):
                return 'sell'
        for pattern in self.add_patterns:
            if pattern.search(text_lower):
                return 'add'
        return ''
    
    def extract_quantity(self, text: str) -> int:
        """Extract quantity from text. Default to 1."""
        match = re.search(r'\b(\d+)\b', text)
        if match:
            return max(int(match.group(1)), 1)
        return 1
    
    def extract_shop(self, text: str) -> Optional[str]:
        """Extract shop name from text."""
        text_lower = text.lower()
        for shop in SHOPS:
            if shop in text_lower:
                return shop
        return 'Main Store'  # Default
    
    def extract_category(self, text: str) -> Optional[str]:
        """Extract category from text."""
        text_lower = text.lower()
        for category in CATEGORIES:
            if category in text_lower:
                return category
        return None
    
    def predict(self, text: str) -> Dict[str, Any]:
        """Predict intent and extract all fields. Returns JSON-serializable dict."""
        intent = self.extract_intent(text)
        return {
            'intent': intent,
            'quantity': self.extract_quantity(text),
            'shop': self.extract_shop(text),
            'category': self.extract_category(text),
            'confidence': 0.95 if intent else 0.0,
        }

# Initialize model
model = IntentModel()
print('✓ Model loaded')

In [ ]:
# Test the model
test_cases = [
    'sell 10 electronics',
    'add 5 peripherals',
    'maine abhi 10 abc bech diye hai',
    'yahan 20 workstations store kar do',
]

for test in test_cases:
    result = model.predict(test)
    print(f'Input: {test}')
    print(f'Output: {json.dumps(result, indent=2)}')
    print()

In [ ]:
# Export model for backend integration
import pickle
import os

model_path = '/Users/dheerajprajapati/Documents/shopai/backend/dashboard/services/intent_model.pkl'
os.makedirs(os.path.dirname(model_path), exist_ok=True)

with open(model_path, 'wb') as f:
    pickle.dump(model, f)

print(f'✓ Model saved to {model_path}')